# BVC Market Data Pipeline

Fetch and analyze live Moroccan stock market data from the **Bourse de Casablanca**.

This notebook covers:
- Live stock prices (all listed instruments)
- Index values (MASI, MADEX, CFG25…)
- Top gainers / losers / most active
- Market summary statistics
- Storing BVC data in PostgreSQL for the RAG pipeline

---

In [1]:
import sys
import os
from pathlib import Path
from _setup import PROJECT_ROOT

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\lenovo\Desktop\Pfe 2026\financial-advisor


## 1. Fetch Live Market Data

In [2]:
from retrieval.bvc_scraper import get_market_live, get_market_summary

# Fetch all listed instruments
stocks = get_market_live()
print(f"Fetched {len(stocks)} instruments from Casablanca Bourse\n")

# Display as DataFrame
df_stocks = pd.DataFrame(stocks)
df_stocks.head(10)

Fetched 112 instruments from Casablanca Bourse



,ticker,name,sector,last_price,ref_price,open,high,low,variation_pct,volume,qty_traded,nb_trades,capitalisation,status,bid_price,ask_price
0,ADH,DOUJA PROM ADDOHA,Participation et promotion immobilières,34.2500000000,34.8900000000,34.2000000000,34.9600000000,34.1400000000,-1.8343364861,83656.2200000000,2434.0000000000,20,13787380449.5000000000,-,34.2500000000,34.9000000000
1,ADI,ALLIANCES,Participation et promotion immobilières,398.0000000000,390.0000000000,395.0000000000,398.0000000000,395.0000000000,2.0512820513,42271.0000000000,107.0000000000,5,8787278024.0000000000,-,395.0000000000,397.5000000000
2,ATL,ATLANTASANAD,Assurances,126.2000000000,126.3500000000,127.8000000000,127.9000000000,126.2000000000,-0.1187178472,50404.4000000000,399.0000000000,13,7607789689.0000000000,-,126.2500000000,126.5000000000
3,ATW,ATTIJARIWAFA BANK,Banques,690.0000000000,685.2000000000,685.0000000000,690.0000000000,682.2000000000,0.7005253940,1530159.1000000000,2223.0000000000,49,148447178910.0000000000,-,690.0000000000,692.0000000000
4,BCP,BCP,Banques,266.0000000000,268.9500000000,266.0000000000,266.0000000000,261.5000000000,-1.0968581521,1295446.4500000000,4903.0000000000,69,54081117818.0000000000,-,266.0000000000,268.5000000000
5,CIH,CIH,Banques,346.0000000000,346.0000000000,341.0500000000,348.9500000000,341.0000000000,0.0000000000,54585.4500000000,160.0000000000,4,12319545904.0000000000,-,341.1000000000,347.9000000000
6,CSR,COSUMAR,Agroalimentaire et Production,194.3500000000,195.2500000000,195.2500000000,198.0000000000,194.3500000000,-0.4609475032,107276.4500000000,549.0000000000,22,18363576242.0500000000,-,194.6000000000,195.9000000000
7,DHO,DELTA HOLDING,Sociétés de Portefeuilles / Holdings,60.0000000000,60.2000000000,58.5000000000,60.0000000000,58.5000000000,-0.3322259136,6615.0200000000,112.0000000000,5,5256000000.0000000000,-,59.0000000000,60.0000000000
8,IAM,ITISSALAT AL-MAGHRIB,Télécommunications,92.2000000000,91.9000000000,92.0000000000,92.2000000000,91.9000000000,0.3264417845,722051.7000000000,7848.0000000000,28,81052590348.0000000000,-,91.9000000000,92.2000000000
9,LES,LESIEUR CRISTAL,Agroalimentaire et Production,350.1500000000,370.0000000000,350.1500000000,350.1500000000,350.1500000000,-5.3648648649,350.1500000000,1.0000000000,1,9675173226.5000000000,-,350.2500000000,378.0000000000


In [3]:
# Market summary
summary = get_market_summary()
print("Market Summary")
print("=" * 40)
for key, val in summary.items():
    print(f"  {key:30s}: {val:>15,}" if isinstance(val, (int, float)) else f"  {key:30s}: {val}")

Market Summary
  total_instruments             :             112
  gainers                       :              24
  losers                        :              28
  stable                        :              60
  total_volume_mad              :   35,904,100.33
  total_capitalisation_mad      : 1,080,131,193,132.59


## 2. Indices (MASI, MADEX, CFG25…)

In [4]:
from retrieval.bvc_scraper import get_indices

indices = get_indices()
df_indices = pd.DataFrame(indices)
print(f"Fetched {len(indices)} indices\n")
df_indices

Fetched 34 indices



,category,name,code,value,previous,variation_pct,variation_ytd,high,low,capitalisation
0,Principaux indices,MASI,MASI,18198.6260000000,18121.2741000000,0.4268568511,-3.44,18203.6911000000,18086.1932000000,271142861072.6600000000
1,Principaux indices,MASI 20,MSI20,1336.2974410000,1333.5221720000,0.2081156998,-10.05,1336.7871610000,1329.8416910000,180737492723.5700000000
2,Principaux indices,MASI ESG,ESGI,1299.2338530000,1286.4624850000,0.9927509079,3.81,1301.7374730000,1285.8593050000,32809887833.7510000000
3,Principaux indices,MASI Mid and Small Cap,MASIMS,1829.5344290000,1829.0757530000,0.0250769275,-0.65,1836.8535000000,1825.6211510000,44044739667.2200000000
4,Indices sectoriels,MASI AGROALIMENTAIRE / PRODUCTION,AGRO,33923.4675000000,33923.4675000000,0.0000000000,-2.05,33923.4675000000,33923.4675000000,10667686490.5250000000
5,Indices sectoriels,MASI ASSURANCES,ASSUR,6689.2264000000,6689.2264000000,0.0000000000,10.55,6689.2264000000,6689.2264000000,8652491771.6500000000
6,Indices sectoriels,MASI BANQUES,BANK,17790.2673000000,17790.2673000000,0.0000000000,-8.79,17790.2673000000,17790.2673000000,76462416090.6050000000
7,Indices sectoriels,MASI BATIMENT ET MATERIAUX DE CONSTRUCTION,B&MC,27942.1212000000,27942.1212000000,0.0000000000,-12.24,27942.1212000000,27942.1212000000,48283962733.7150000000
8,Indices sectoriels,MASI BOISSONS,BOISS,12940.6762000000,12940.6762000000,0.0000000000,-5.37,12940.6762000000,12940.6762000000,1558625859.4000000000
9,Indices sectoriels,MASI CHIMIE,CHIM,3068.2528000000,3068.2528000000,0.0000000000,-29.75,3068.2528000000,3068.2528000000,359153437.5000000000


## 3. Top Gainers / Losers / Most Active

In [5]:
from retrieval.bvc_scraper import get_top_gainers, get_top_losers, get_most_active

print("TOP GAINERS")
print("-" * 60)
gainers = get_top_gainers(5)
for s in gainers:
    print(f"  {s['ticker']:8s} {s['name']:30s} +{s['variation_pct']}%  @ {s['last_price']}")

print("\nTOP LOSERS")
print("-" * 60)
losers = get_top_losers(5)
for s in losers:
    print(f"  {s['ticker']:8s} {s['name']:30s} {s['variation_pct']}%  @ {s['last_price']}")

print("\nMOST ACTIVE (by volume)")
print("-" * 60)
active = get_most_active(5)
for s in active:
    print(f"  {s['ticker']:8s} {s['name']:30s} Vol: {s['volume']}")

TOP GAINERS
------------------------------------------------------------
  STR      STROC INDUSTRIE                +4.1274238227%  @ 187.9500000000
  CDM      CDM                            +3.7113402062%  @ 1006.0000000000
  CMG      CMGP GROUP                     +3.5671641791%  @ 346.9500000000
  MDP      MED PAPER                      +2.9200000000%  @ 25.7300000000
  FBR      FENIE BROSSETTE                +2.1603808129%  @ 279.0000000000

TOP LOSERS
------------------------------------------------------------
  S2M      S.M MONETIQUE                  -9.0661478599%  @ 467.4000000000
  CRS      CARTIER SAADA                  -7.6167076167%  @ 26.3200000000
  SOT      SOTHEMA                        -6.9512862616%  @ 340.0000000000
  LES      LESIEUR CRISTAL                -5.3648648649%  @ 350.1500000000
  AFI      AFRIC INDUSTRIES SA            -4.7337278107%  @ 322.0000000000

MOST ACTIVE (by volume)
------------------------------------------------------------
  MSA      SODEP-Ma

## 4. Search for a Specific Stock

In [6]:
from retrieval.bvc_scraper import get_stock_by_ticker

# Try a well-known Moroccan stock
ticker = "ATW"  # Attijariwafa Bank
stock = get_stock_by_ticker(ticker)

if stock:
    print(f"\n{stock['name']} ({stock['ticker']})")
    print(f"   Sector:      {stock['sector']}")
    print(f"   Last Price:  {stock['last_price']} MAD")
    print(f"   Variation:   {stock['variation_pct']}%")
    print(f"   Open:        {stock['open']}")
    print(f"   High:        {stock['high']}")
    print(f"   Low:         {stock['low']}")
    print(f"   Volume:      {stock['volume']}")
else:
    print(f"Ticker '{ticker}' not found in live data.")


ATTIJARIWAFA BANK (ATW)
   Sector:      Banques
   Last Price:  690.0000000000 MAD
   Variation:   0.7005253940%
   Open:        685.0000000000
   High:        690.0000000000
   Low:         682.2000000000
   Volume:      1531539.1000000000


## 5. Store BVC Data in PostgreSQL

Stores the live data as OHLCV rows so the SQL route can answer queries about Moroccan stocks.

In [7]:
from datetime import datetime
from storage.relational_db import RelationalDB

def parse_bvc_float(val):
    """Parse BVC string numbers to float."""
    if val is None or str(val).strip() in ("", "-", "nan"):
        return None
    try:
        return float(str(val).replace(",", ".").replace(" ", "").replace("\xa0", ""))
    except (ValueError, AttributeError):
        return None

# Build records
today = datetime.now().date()
records = []
for s in stocks:
    ticker = (s.get("ticker") or "").strip().upper()
    close = parse_bvc_float(s.get("last_price"))
    if not ticker or close is None:
        continue
    records.append({
        "ticker": f"BVC.{ticker}",
        "date": today,
        "open": parse_bvc_float(s.get("open")),
        "high": parse_bvc_float(s.get("high")),
        "low": parse_bvc_float(s.get("low")),
        "close": close,
        "volume": int(parse_bvc_float(s.get("volume")) or 0),
    })

print(f"Prepared {len(records)} BVC price records")

# Upsert into PostgreSQL
db = RelationalDB()
db.setup_database()
inserted = db.insert_prices(records)
print(f"[OK] Upserted {inserted} rows into stock_prices")

Prepared 112 BVC price records
[OK] Upserted 112 rows into stock_prices


In [8]:
# Verify: query some BVC tickers from the DB
sample = db.execute_sql(
    "SELECT ticker, close, volume, date FROM stock_prices "
    "WHERE ticker LIKE 'BVC.%' ORDER BY ticker LIMIT 10"
)
pd.DataFrame(sample)

,ticker,close,volume,date
0,BVC.AADHA,35.50,0,2026-06-19
1,BVC.AADHA,34.25,0,2026-07-02
2,BVC.AATHA,111.65,0,2026-06-19
3,BVC.AATHA,121.65,0,2026-07-02
4,BVC.AATHB,67.00,0,2026-06-19
5,BVC.AATHB,73.00,0,2026-07-02
6,BVC.ABCIA,124.80,0,2026-06-19
7,BVC.ABCIA,125.00,0,2026-07-02
8,BVC.ABOAA,54.85,0,2026-06-19
9,BVC.ABOAA,56.27,0,2026-07-02
